# AP2526 Concurrency 2

Topics covered in this module:  
- Communication between threads and processes.
- Locks

**Warning**: Under Windows (and perhaps other operating systems as well), when running multiple processes each process uses its own console to output printed information, and usually only the console of the main process is visible. Moreover, multiprocessing has severe issues in notebooks under Windows. This is because of the Jupyter iPython interpreter, and the fact that Windows does not support all features needed to implement good multiprocessing. Therefore, in this notebook all code was written with threads, not processes. For processes it would not have been much different.

This notebook contains basic information on concurrency and formative exercises. Use it together with other materials provided by the course, such as the book, the sheets, and example code.

---

## Race Condition

One of the problems that might arise when you attempt to make a program run concurrently is a race condition. This condition occurs when you need threads or processes to run in a certain order or after certain events have taken place, but they do not do so due to lack of control. 
In many cases, this happens when access to a variable or to a shared resource is not handled properly.

---

## Locks (Mutex Locks)

A Lock is an object that is used to make sure only one thread or process can access a shared resource (which can be a variable, a database, a web service, etc.) at a time. Locks are placed in the code of the thread or process that is using the resource. 

For threads, create a lock, after `import threading`, with the command: `<lock> = threading.Lock()`<br>
Acquire the lock with `<lock>.acquire()`<br>
Release the lock with `<lock>.release()`<br>
You may also write your code under `with <lock>:`, which will then acquire the lock at the block's start and release it at the end.

Locks can also be used with `multiprocessing` (just use `multiprocessing` instead of `threading`).

Example:

In [3]:
# NB2_01.py

import threading, time

lock = threading.Lock()
counter = 0

def add_up( max_add ):
    global counter
    for _ in range( max_add ):
        with lock:
            c = counter + 1
            time.sleep(1/max_add)
            counter = c
            print( counter )

t1 = threading.Thread( target=add_up, args=[5] ) 
t2 = threading.Thread( target=add_up, args=[10] ) 
t1.start()
t2.start()
t1.join()
t2.join()
print( "end" )


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
end


### Formative Exercise 1

The following program uses two threads to get numbers from a list and add them up in a variable `total`. If this program would work well, it would end up with a total of 45 and it wouldn't give an error. However, in general it will end up with the right total, but the program will produce an error.

Fix it, using a lock.

In [4]:
# NB2_02.py

import threading, time

numlist = list( range(10) )
total = 0
lock = threading.Lock()

### Exercise 1 ###############################

def process():
    global total
    global numlist
    while True:
        lock.acquire()
        if len( numlist ) <= 0:
            lock.release()
            return
        total += numlist[0]
        time.sleep(0.1)
        del numlist[0]
        lock.release()

##############################################

tr1 = threading.Thread( target=process, args=[] )
tr2 = threading.Thread( target=process, args=[] )
tr1.start()
tr2.start()
tr1.join()
tr2.join()
print( total )
print( 'end' )

45
end


---

## RLock (Re-entrant Mutex Lock)

RLocks are quite the same as Locks, but can be used by one thread multiple times. Usage is the same as `Lock()`, except that you use `RLock()`.

### Formative Exercise 2

The provided code below aims to implement a concurrent solution for a program that needs to perform some tasks (simulated with the function `task()`) and report on each of them (simulated with the function `report()`). In order to make sure the program is thread-safe, the programmer has implemented a
Lock. However, this implementation of thread-safety is blocking the program: if you run it, you will find that it gets stuck.

Looking at the code, find out what is the reason for this behavior. Adjust the code such that thread-safety is handled in a way that allows the program to function normally. The `task()` and `report()` functions should keep the same functionality, and protect the same lines of code that are protected right now. Do not merge these functions.


In [ ]:
# NB2_03.py

### Exercise 2 ###############################

from time import sleep
from random import random
from threading import *

def report( lock, identifier ):
    with lock:
        sleep(1)
        print( f"Process {identifier} reported" )

def task( lock, identifier, value ):
    with lock:
        print( f"Process {identifier} running... " )
        sleep(value)
        report( lock, identifier )

def main():
    lock = RLock()
    threads = [Thread( target=task, args=(lock, i, random()) ) for i in range(10)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

if __name__ == "__main__":
    main()

##############################################


Process 0 running... 
Process 0 reported
Process 1 running... 
Process 1 reported
Process 2 running... 
Process 2 reported
Process 3 running... 


---
## Semaphores

A semaphore is a lock that can be acquired by a certain limited number of threads or processes. Once the maximum number is reached, new threads or processes that want to acquire the semaphore have to wait until a spot opens up.

Create a sempahore with `<semaphore> = threading.Semaphore(<limit>)` or `<semaphore> = multiprocessing.Semaphore(<limit>)`. Use `with <semaphore>:` to give threads or processes access to the semaphore.

### Formative Exercise 3

16 players are waiting to use a table tennis table. Only two can play simultaneously. In the code below they will all start playing at about the same time. Change the code, using a semaphore, in such a way that at most two players will be at the table, but in the end all of them will have played.

In [25]:
# NB2_04.py

import threading, time
maxi = threading.Semaphore(2)
### Exercise 3 ###############################

def playTennis( num ):  
    with maxi:
        print( f"Player {num} is playing" )
        time.sleep(1) 
        print( f"Player {num} is leaving the table" )

##############################################

players = []
for x in range(16):
    players += [threading.Thread( target=playTennis, args=(x,) )]
for player in players:
    player.start()
for player in players:
    player.join()
print( "end" )


Player 0 is playing
Player 1 is playing
Player 0 is leaving the tablePlayer 1 is leaving the table

Player 2 is playing
Player 3 is playing
Player 2 is leaving the tablePlayer 3 is leaving the table
Player 4 is playing

Player 5 is playing
Player 4 is leaving the tablePlayer 5 is leaving the table
Player 6 is playing

Player 7 is playing
Player 6 is leaving the tablePlayer 7 is leaving the table
Player 8 is playing

Player 9 is playing
Player 8 is leaving the tablePlayer 9 is leaving the table
Player 10 is playing

Player 11 is playing
Player 11 is leaving the tablePlayer 10 is leaving the table

Player 12 is playing
Player 13 is playing
Player 12 is leaving the tablePlayer 13 is leaving the table
Player 14 is playing

Player 15 is playing
Player 14 is leaving the table
Player 15 is leaving the table
end


---

## Events

Events are used to send a signal from one thread to another. Create an event with `<event> = threading.Event()`. The event is `False` to start with. With `<event>.wait()`, a thread waits for the event to become `True`. With `<event>.set()`, the event is set to `True`. With `<event>.clear()`, the event is set to False.

### Formative Exercise 4

In the example below, a main thread starts multiple subtasks. Then the main thread needs to do some processing. Once it is finished, it should tell the subtasks that they can complete their work. In the implementation below, however, the subtasks will not wait for the main process to tell them to continue. You can use an event to signal to them that they may continue their work. Change the code to that effect. 

In [29]:
# NB2_05.py

import threading, time

### Exercise 4 ###############################
ev = threading.Event()
def subtask( i ):
    print( f"Thread {i} started..." )

    # ...the subtask can do anything here, and then wait for a signal to continue
    
    print( "Waiting for the signal..." )
    # ...should receive a signal
    ev.set()
    print( "Signal received!" )
    # ...the thread can now use the shared resource
    ev.wait()
    print( f"Thread {i} continues..." )
    print( f"Thread {i} was completed" )

def task( nsubtasks=3 ):
    pool=[]
    for i in range( nsubtasks ):
        pool.append( threading.Thread( target=subtask, args=(i,)) )
    for i in range( nsubtasks ):
        pool[i].start()
    print( 'The main task is running now...' )
    time.sleep(3) 
    print( 'The main task is finished. Sending a signal for subtasks to proceed...' )
    for i in range( nsubtasks ):
        pool[i].join()

##############################################
    
if __name__ == "__main__":
    task( nsubtasks=4 )


Thread 0 started...
Waiting for the signal...
Signal received!
Thread 0 continues...
Thread 0 was completed
Thread 1 started...
Waiting for the signal...
Signal received!
Thread 1 continues...
Thread 1 was completed
Thread 2 started...
Waiting for the signal...
Signal received!
Thread 2 continues...
Thread 2 was completed
Thread 3 started...
Waiting for the signal...
Signal received!
Thread 3 continues...
Thread 3 was completed
The main task is running now...
The main task is finished. Sending a signal for subtasks to proceed...


---

## Deadlock

A deadlock occurs when threads or processes are waiting on each other to complete something, causing them to wait forver.

In the example below two threads are "eating". Each of them needs a knife and a fork. One thread takes the knife first, and the other takes the fork first. This causes a deadlock.

You may run this code, but the program will hang. You will have to interrupt execution (in a notebook, this means restarting the kernel). 

In [30]:
# NB2_06.py

#WARNING: Since this example recreates a deadlock, 
#you will have to interrupt the execution.

import threading, time

fork = threading.Lock()
knife = threading.Lock()

def eating1():
    with knife:
        time.sleep(3)
        with fork:
            print( "Much Munch Munch" )
            
def eating2():
    with fork:
        time.sleep(3)
        with knife: 
            print( "Chomp Chomp Chomp" )
            
print( "start" )
t1 = threading.Thread( target=eating1 )
t2 = threading.Thread( target=eating2 )
t1.start()
t2.start()
t1.join()
t2.join()
print( "end" )

start


KeyboardInterrupt: 

### Formative Exercise 5

To prevent deadlocks, try not to have two mutex locks open in the same thread at the same time, or have a consistent ordering for the locks so that they are always acquired in the same order. Solve the problem in the code above (note that the two threads cannot be eating simultaneously).

In [ ]:
# NB2_07.py

import threading, time

fork = threading.Lock()
knife = threading.Lock()

### Exercise 5 ###############################


def eating1():
    with knife:
        time.sleep(3)
        
        with fork:
            print( "Much Munch Munch" )
                   
def eating2():
    with knife:
        time.sleep(3)
       
        with fork:
            print( "Chomp Chomp Chomp" )
          
        

##############################################

print( "start" )
t1 = threading.Thread( target=eating1 )
t2 = threading.Thread( target=eating2 )
t1.start()
t2.start()
t1.join()
t2.join()
print( "end" )

start
Much Munch Munch
Chomp Chomp Chomp
end


---

## Dining Philosophers

In the Dining Philosophers problem, multiple philosophers are sitting around a table, with a fork between each adjoining pair. To eat, they have to grab the two forks next to them, and put them down when they have finished eating. As explained in the lecture, this may lead to a deadlock, depending on when and how the threads which represent the philosophers interleave. 

### Formative Exercise 6

The code below implements the problem for three philosophers. This code will usually (but not always) get into a deadlock where all philosophers are holding a left fork and are waiting for one of the others to put down the fork they are holding. Resolve the problem in this code.

In [3]:
# NB2_08.py

import threading as tr, time, random
lock_lower = tr.Lock()
lock_higher = tr.Lock()
COUNT = 3
ROUNDS = 10
fork = []
for i in range( COUNT ):
    fork.append( tr.Lock() )

### Exercise 6 ###############################

def philosopher( num ):
    first = (num-1)%COUNT
    second = num
    first_fork= min(first, second)
    second_fork = max(first, second)
  
    for i in range( ROUNDS ):
        print( f"Philosopher {num} is thinking..." )
        time.sleep( random.random() )
        with fork[first_fork]:
            print( f"Philosopher {num} takes {first} fork." )
            time.sleep( random.random()/3 )
            with fork[second_fork]:
                print( f"Philosopher {num} takes {second} fork." )
                print( f"Philosopher {num} is eating..." )
                time.sleep( random.random() )
                print( f"Philosopher {num} puts down {second} fork." )
            print( f"Philosopher {num} puts down {first} fork." )
    print( f"Philosopher {num} says: \"Eureka!\"" )
    
##############################################    
    
if __name__ == "__main__":
    threads = [ tr.Thread( target=philosopher, name=f'T{name}', args=(name,)) for name in range(COUNT) ]
    for t in threads:
        t.start()

Philosopher 0 is thinking...
Philosopher 1 is thinking...
Philosopher 2 is thinking...


Philosopher 1 takes 0 fork.
Philosopher 1 takes 1 fork.
Philosopher 1 is eating...
Philosopher 1 puts down 1 fork.
Philosopher 1 puts down 0 fork.
Philosopher 1 is thinking...
Philosopher 2 takes 1 fork.
Philosopher 2 takes 2 fork.
Philosopher 2 is eating...
Philosopher 0 takes 2 fork.
Philosopher 2 puts down 2 fork.
Philosopher 2 puts down 1 fork.
Philosopher 2 is thinking...
Philosopher 0 takes 0 fork.
Philosopher 0 is eating...
Philosopher 2 takes 1 fork.
Philosopher 0 puts down 0 fork.
Philosopher 0 puts down 2 fork.
Philosopher 0 is thinking...
Philosopher 1 takes 0 fork.
Philosopher 2 takes 2 fork.
Philosopher 2 is eating...
Philosopher 2 puts down 2 fork.
Philosopher 2 puts down 1 fork.
Philosopher 2 is thinking...
Philosopher 1 takes 1 fork.
Philosopher 1 is eating...
Philosopher 1 puts down 1 fork.
Philosopher 1 puts down 0 fork.
Philosopher 1 is thinking...
Philosopher 0 takes 2 fork.
Philosopher 2 takes 1 fork.
Philosopher 2 takes 2 fork.
Philosopher 2 is eating...
Philosoph